 environment config

In [18]:
# 请将此代码块粘贴到Jupyter Notebook的第一个代码单元格中
# 单元格 1: 导入与环境设置
import pandas as pd
import json
import re
import emoji
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 下载必要的NLTK数据 (如果尚未下载)
try:
    word_tokenize("test string") # 测试punkt是否可用
except LookupError:
    print("NLTK 'punkt' not found. Downloading...")
    nltk.download('punkt', quiet=True)
    print("'punkt' downloaded.")

try:
    stopwords.words('english') # 测试stopwords是否可用
except LookupError:
    print("NLTK 'stopwords' not found. Downloading...")
    nltk.download('stopwords', quiet=True)
    print("'stopwords' downloaded.")

print("必要的库已导入，NLTK资源已检查/下载完毕。")

NLTK 'punkt' not found. Downloading...
'punkt' downloaded.
必要的库已导入，NLTK资源已检查/下载完毕。


# ## 1. 加载数据
# 定义一个函数来加载JSON格式的数据。

In [19]:
# 请将此代码块粘贴到Jupyter Notebook的第三个代码单元格中
# 单元格 3: load_json_data 函数
def load_json_data(file_path):
    """加载JSON文件"""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

print("load_json_data 函数已定义。")

load_json_data 函数已定义。


# ## 2. 清理Tweet文本
# 定义一个函数来清理单个tweet文本。包括：
# - 转换为小写
# - 移除URL
# - 移除@提及
# - 移除#标签符号 (保留标签文本)
# - 移除多余的空格

In [20]:
# 单元格 5: clean_tweet 函数 (已更新，增加emoji和非指定语言字符的清理)
import emoji
import re

def clean_tweet(text):
    """
    清理单个tweet文本。
    1. 转换为小写
    2. 移除URL (已在函数中，再次确认)
    3. 移除@提及
    4. 移除#标签符号
    5. 移除Emoji
    6. 移除任何非拉丁字母、非数字、非基本标点的字符
    7. 移除多余空格
    """
    # 确保输入是字符串并转为小写
    text = str(text).lower() 
    
    # 1. 移除URL (重复检查，以防万一)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 2. 移除@提及
    text = re.sub(r'@\w+', '', text)
    
    # 3. 移除#标签符号（保留标签文本）
    text = re.sub(r'#', '', text)
    
    # 4. 移除Emoji
    # 首先将emoji转换为唯一的文本标记，然后移除这些标记
    # 这样做比直接用正则表达式移除更准确
    text = emoji.demojize(text, delimiters=(" emoji_", "_emoji "))
    text = re.sub(r' emoji_.*?_emoji ', ' ', text)
    
    # 5. 移除所有非拉丁字母、非数字、非基本标点和非空格的字符
    # 这个正则表达式保留了:
    # a-z (英文小写字母)
    # 0-9 (数字)
    # \s (空格类字符)
    # áéíóúñü (西班牙语和部分其他拉丁语言特有字符)
    # .!?, (一些基本标点)
    # [^...] 表示匹配不在集合内的任何字符，我们用它来找到并替换掉所有“不想要的”字符
    text = re.sub(r'[^a-z0-9\s.!?,\u00E1\u00E9\u00ED\u00F3\u00FA\u00F1\u00FC]', '', text)
    
    # 6. 移除多余的空格
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("clean_tweet 函数已更新，增加了移除emoji和非拉丁语系字符的功能。")

clean_tweet 函数已更新，增加了移除emoji和非拉丁语系字符的功能。


# ## 3. 处理整个数据集
# 定义一个主函数来加载数据，将其转换为DataFrame，并应用清理函数。

In [21]:
# 单元格 7: process_data 函数 (已更新)
def process_data(file_path):
    """处理整个数据集"""
    # 读取数据
    raw_data = load_json_data(file_path)
    
    tweets_list = [] # 初始化一个空列表来收集推文对象
    
    # 检查raw_data是否为字典
    if isinstance(raw_data, dict):
        # 如果是字典，我们假设它的值是推文对象
        # 我们将把这些值收集到一个列表中
        tweets_list = list(raw_data.values())
        if not tweets_list:
             raise ValueError("JSON顶层字典的值为空，无法提取推文列表。")
        print(f"从顶层字典的值中提取了 {len(tweets_list)} 个条目作为推文。")
    elif isinstance(raw_data, list): 
        # 以防万一，如果顶层直接是列表 (尽管根据调试输出不太可能)
        tweets_list = raw_data
        if not tweets_list:
            raise ValueError("JSON顶层列表为空。")
        print("JSON顶层直接是一个列表，将其用作推文列表。")
    else:
        raise ValueError(f"无法识别的JSON数据结构。顶层既不是预期的字典也不是列表，而是类型: {type(raw_data)}")
            
    if not tweets_list: # 双重检查
        raise ValueError("未能从JSON数据中成功提取推文列表。")
            
    # 现在 tweets_list 应该是一个包含推文对象的列表了
    # 每个推文对象应该是一个字典
    df = pd.DataFrame(tweets_list)
    
    text_column_to_use = None
    # 根据EXIST数据集的常见结构，推文文本可能在 'tweet', 'text', 'text_es', 'text_en'等键中
    # 我们需要从DataFrame的列中找到包含实际文本的列
    possible_text_columns = ['tweet', 'text', 'text_es', 'text_en', 'tweet_text', 'full_text'] 
    
    print(f"\nDataFrame的列名: {list(df.columns)}")

    for col in possible_text_columns:
        if col in df.columns:
            text_column_to_use = col
            print(f"使用列 '{text_column_to_use}' 作为推文文本。")
            break # 找到第一个匹配的就使用
            
    if text_column_to_use is None:
        # 如果在常见列名中找不到，提示用户检查并可能需要手动指定
        raise ValueError(f"DataFrame中未找到合适的文本列 (已尝试: {possible_text_columns})。" \
                         f" 请检查DataFrame的列: {list(df.columns)} 并更新possible_text_columns。")
            
    tqdm.pandas(desc="清理推文")
    # 确保要清理的列中的所有值都是字符串，以避免 .lower() 等操作的错误
    df[text_column_to_use] = df[text_column_to_use].astype(str)
    df['cleaned_text'] = df[text_column_to_use].progress_apply(clean_tweet)
    
    return df, text_column_to_use

print("process_data 函数已更新。")

process_data 函数已更新。


In [22]:
# 调试单元格：检查JSON结构
debug_file_path = "/Users/zenanjin/Study/德语TUM/sexism/nlp_practical_2025_sEXism/2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json"
raw_data_debug = None
print(f"正在尝试加载调试文件: {debug_file_path}")
try:
    with open(debug_file_path, 'r', encoding='utf-8') as f:
        raw_data_debug = json.load(f)
    
    print("文件加载成功。")

    if isinstance(raw_data_debug, dict):
        print("JSON顶层是一个字典。以下是它的键 (keys):")
        print(list(raw_data_debug.keys()))
        
        print("\n详细检查每个键对应的值:")
        for key, value in raw_data_debug.items():
            print(f"--- 键: '{key}' ---")
            print(f"  值的类型: {type(value)}")
            if isinstance(value, list):
                print(f"  这是一个列表，其长度为: {len(value)}")
                if len(value) > 0:
                    print(f"  列表的前1个元素是:")
                    item = value[0]
                    print(f"    元素 0 类型: {type(item)}")
                    if isinstance(item, dict):
                        print(f"    元素 0 是一个字典，其键为: {list(item.keys())}")
                        # 尝试打印可能包含文本的键
                        possible_text_keys_in_item = ['tweet', 'text', 'text_es', 'text_en', 'content', 'message']
                        for p_key in possible_text_keys_in_item:
                            if p_key in item:
                                print(f"      预览 '{p_key}': {str(item[p_key])[:100]}...")
                                break
                    else:
                        print(f"    元素 0 (前100字符): {str(item)[:100]}...")
                else:
                    print("  列表为空。")
            elif isinstance(value, dict):
                 print(f"  这是一个字典，其键为: {list(value.keys())}")
                 # 如果嵌套字典不深，可以考虑打印其内容，但要注意避免过长输出
                 # print(f"    嵌套字典内容 (部分): {str(value)[:200]}...")
            else:
                # 对于非列表、非字典的值，直接打印一部分
                print(f"  值 (前100字符): {str(value)[:100]}...")
            print("-" * 20)

    elif isinstance(raw_data_debug, list):
        print("JSON顶层是一个列表。")
        print(f"列表长度为: {len(raw_data_debug)}")
        if len(raw_data_debug) > 0:
            print(f"列表的前1个元素是:")
            item = raw_data_debug[0]
            print(f"  元素 0 类型: {type(item)}")
            if isinstance(item, dict):
                print(f"  元素 0 是一个字典，其键为: {list(item.keys())}")
                possible_text_keys_in_item = ['tweet', 'text', 'text_es', 'text_en', 'content', 'message']
                for p_key in possible_text_keys_in_item:
                    if p_key in item:
                        print(f"      预览 '{p_key}': {str(item[p_key])[:100]}...")
                        break
            else:
                print(f"  元素 0 (前100字符): {str(item)[:100]}...")
        else:
            print("列表为空。")
    else:
        print(f"JSON顶层既不是字典也不是列表，而是类型: {type(raw_data_debug)}")

except FileNotFoundError:
    print(f"错误：找不到文件 {debug_file_path}")
except json.JSONDecodeError as e:
    print(f"错误：JSON文件解码失败 - {e}")
except Exception as e:
    import traceback
    print(f"发生未知错误: {e}")
    print(traceback.format_exc())

正在尝试加载调试文件: /Users/zenanjin/Study/德语TUM/sexism/nlp_practical_2025_sEXism/2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json
文件加载成功。
JSON顶层是一个字典。以下是它的键 (keys):
['100001', '100002', '100003', '100004', '100005', '100006', '100007', '100008', '100009', '100010', '100011', '100012', '100013', '100014', '100015', '100016', '100017', '100018', '100019', '100020', '100021', '100022', '100023', '100024', '100025', '100026', '100027', '100028', '100029', '100030', '100031', '100032', '100033', '100034', '100035', '100036', '100037', '100038', '100039', '100040', '100041', '100042', '100043', '100044', '100045', '100046', '100047', '100048', '100049', '100050', '100051', '100052', '100053', '100054', '100055', '100056', '100057', '100058', '100059', '100060', '100061', '100062', '100063', '100064', '100065', '100066', '100067', '100068', '100069', '100070', '100071', '100072', '100073', '100074', '100075', '100076', '100077', '100078', '100079', '100080', '100081', '100082', '

In [23]:
# 请将此代码块粘贴到Jupyter Notebook的第九个代码单元格中
# 单元格 9: 运行数据清理并查看结果
# 定义训练数据文件路径
# 请确保路径相对于notebook或者使用绝对路径
train_file_path = "/Users/zenanjin/Study/德语TUM/sexism/nlp_practical_2025_sEXism/2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json"
# train_file_path = "/Users/zenanjin/Study/德语TUM/sexism/nlp_practical_2025_sEXism/2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json" # 绝对路径示例

train_df = None 
original_text_column_name = None 

# 处理训练数据
try:
    train_df, original_text_column_name = process_data(train_file_path)
    
    print("\n数据集基本信息：")
    train_df.info()
    
    if original_text_column_name:
        print("\n数据示例 (前5条)：")
        print(train_df[[original_text_column_name, 'cleaned_text']].head())
    else:
        print("\n警告: 未能确定原始文本列名，无法显示清理前后对比。")
        print(train_df[['cleaned_text']].head()) # 只显示清理后的文本
    
    empty_cleaned_text_count = train_df[train_df['cleaned_text'] == ''].shape[0]
    if empty_cleaned_text_count > 0:
        print(f"\n警告：有 {empty_cleaned_text_count} 条推文在清理后文本内容为空。")
        # print("\n空文本示例：")
        # if original_text_column_name:
        #     print(train_df[train_df['cleaned_text'] == ''][[original_text_column_name, 'cleaned_text']].head())
        # else:
        #     print(train_df[train_df['cleaned_text'] == ''][['cleaned_text']].head())
            
except FileNotFoundError:
    print(f"错误：找不到文件 {train_file_path}。请检查文件路径是否正确。")
except ValueError as e:
    print(f"处理数据时发生值错误: {e}")
except Exception as e:
    import traceback
    print(f"发生未知错误: {e}")
    print(traceback.format_exc())

从顶层字典的值中提取了 6920 个条目作为推文。

DataFrame的列名: ['id_EXIST', 'lang', 'tweet', 'number_annotators', 'annotators', 'gender_annotators', 'age_annotators', 'ethnicities_annotators', 'study_levels_annotators', 'countries_annotators', 'labels_task1_1', 'labels_task1_2', 'labels_task1_3', 'split']
使用列 'tweet' 作为推文文本。


清理推文: 100%|██████████| 6920/6920 [00:00<00:00, 11817.84it/s]


数据集基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6920 entries, 0 to 6919
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   id_EXIST                 6920 non-null   object
 1   lang                     6920 non-null   object
 2   tweet                    6920 non-null   object
 3   number_annotators        6920 non-null   int64 
 4   annotators               6920 non-null   object
 5   gender_annotators        6920 non-null   object
 6   age_annotators           6920 non-null   object
 7   ethnicities_annotators   6920 non-null   object
 8   study_levels_annotators  6920 non-null   object
 9   countries_annotators     6920 non-null   object
 10  labels_task1_1           6920 non-null   object
 11  labels_task1_2           6920 non-null   object
 12  labels_task1_3           6920 non-null   object
 13  split                    6920 non-null   object
 14  cleaned_text             6920 

# ## 5. (可选) 进一步处理和保存数据
# 您可以根据需要添加更多的数据清理步骤，例如：
# - 词形还原 (Lemmatization)
# - 词干提取 (Stemming)
# - 移除停用词 (Stopwords removal) - 下方有示例
# - 处理表情符号 (将表情符号转换为文本或移除)
# 
# 完成清理后，您可以将清理后的数据保存到新的文件中。 - 下方有示例

In [24]:
# 单元格 11: 检查并保存清理后的数据 (保留所有原始列)

# 确保 train_df 已成功创建并且包含 cleaned_text 列
if 'train_df' in locals() and train_df is not None and 'cleaned_text' in train_df.columns:
    output_file_path = "cleaned_EXIST2025_training.csv"
    
    try:
        print("\nDataFrame中的所有列名:")
        print(list(train_df.columns))
        
        print("\n准备保存的DataFrame (前5行，展示部分重要列和清理后的文本):")
        columns_to_preview = []
        if 'id' in train_df.columns:
            columns_to_preview.append('id')
        
        if 'original_text_column_name' in locals() and original_text_column_name in train_df.columns:
            columns_to_preview.append(original_text_column_name)
        elif 'tweet' in train_df.columns:
            columns_to_preview.append('tweet')
        elif 'text' in train_df.columns:
            columns_to_preview.append('text')

        columns_to_preview.append('cleaned_text')

        if 'label' in train_df.columns:
            columns_to_preview.append('label')
        
        columns_to_preview = [col for col in columns_to_preview if col in train_df.columns]
        
        if columns_to_preview:
            print(train_df[columns_to_preview].head())
        else:
            print("无法自动确定要预览的列，打印整个DataFrame的前5行：")
            print(train_df.head())

        # 保存为 utf-8-sig 编码，以便 Excel 正确读取
        train_df.to_csv(output_file_path, index=False, encoding='utf-8-sig')
        print(f"\n清理后的数据已成功保存到: {output_file_path}")
        print(f"完整路径: /Users/zenanjin/Study/德语TUM/sexism/nlp_practical_2025_sEXism/{output_file_path}")

    except Exception as e:
        print(f"\n保存CSV文件时出错: {e}")
        import traceback
        print(traceback.format_exc())

elif 'train_df' in locals() and train_df is None:
    print("\n错误：train_df 是 None。无法保存数据。")
elif 'train_df' not in locals():
    print("\n错误：train_df 变量不存在。请确保已运行前面的数据加载和清理步骤。")
else:
    print(f"\n错误：'cleaned_text' 列不存在于 train_df 中。现有列: {train_df.columns}")

print("\n单元格11执行完毕。")


DataFrame中的所有列名:
['id_EXIST', 'lang', 'tweet', 'number_annotators', 'annotators', 'gender_annotators', 'age_annotators', 'ethnicities_annotators', 'study_levels_annotators', 'countries_annotators', 'labels_task1_1', 'labels_task1_2', 'labels_task1_3', 'split', 'cleaned_text']

准备保存的DataFrame (前5行，展示部分重要列和清理后的文本):
                                               tweet  \
0  @TheChiflis Ignora al otro, es un capullo.El p...   
1  @ultimonomada_ Si comicsgate se parece en algo...   
2  @Steven2897 Lee sobre Gamergate, y como eso ha...   
3  @Lunariita7 Un retraso social bastante lamenta...   
4  @novadragon21 @icep4ck @TvDannyZ Entonces como...   

                                        cleaned_text  
0  ignora al otro, es un capullo.el problema con ...  
1  si comicsgate se parece en algo a gamergate pu...  
2  lee sobre gamergate, y como eso ha cambiado la...  
3  un retraso social bastante lamentable, gamerga...  
4  entonces como así es el mercado lo mejor no es...  

清理后的数据已成功保存到: cl

Data augmentation

In [27]:
# 单元格 12: AEDA 增强
import random

def aeda_augment(text, punctuations=None, ratio=0.3):
    """
    对输入文本执行 AEDA 数据增强：在随机位置插入标点。
    """
    if punctuations is None:
        punctuations = ['.', ',', ';', '!', '?', ':']
    
    words = text.split()
    if len(words) < 2:
        return text  # 不增强过短文本

    n_punc = max(1, int(len(words) * ratio))

    for _ in range(n_punc):
        insert_idx = random.randint(0, len(words))
        punc = random.choice(punctuations)
        words.insert(insert_idx, punc)

    return ' '.join(words)

# 应用 AEDA 增强
if 'train_df' in locals() and 'cleaned_text' in train_df.columns:
    tqdm.pandas(desc="AEDA增强中")
    train_df['aug_aeda'] = train_df['cleaned_text'].progress_apply(lambda x: aeda_augment(x, ratio=0.3))

    # 展示部分增强示例
    print("\n原始与AEDA增强对比示例：")
    print(train_df[['cleaned_text', 'aug_aeda']].sample(5))
else:
    print("错误：未找到 train_df 或 'cleaned_text' 列，无法执行 AEDA 增强。")
    
print(train_df.columns)

AEDA增强中: 100%|██████████| 6920/6920 [00:00<00:00, 111204.02it/s]


原始与AEDA增强对比示例：
                                           cleaned_text  \
3340  anda mira fátima la que conducía tan bien como...   
4608  having been in an industry that glorifies this...   
3723  mgtow ka india wing bhi hai kya, if yes then t...   
1615  aveces tengo el autoestima re arriba, aveces m...   
298   lo que hizo ella, muy pocas personas le pagarí...   

                                               aug_aeda  
3340  anda ? mira . : ? fátima la que conducía tan b...  
4608  having ? been in an ; industry that glorifies ...  
3723  mgtow ka india wing , bhi hai ; kya, if , . : ...  
1615  aveces tengo ? el autoestima re , arriba, avec...  
298   lo que hizo . ella, muy pocas ! personas le pa...  
Index(['id_EXIST', 'lang', 'tweet', 'number_annotators', 'annotators',
       'gender_annotators', 'age_annotators', 'ethnicities_annotators',
       'study_levels_annotators', 'countries_annotators', 'labels_task1_1',
       'labels_task1_2', 'labels_task1_3', 'split', 'cleaned_t

In [29]:
# 单元格 13（可选）: 合并原始与AEDA数据
if 'aug_aeda' in train_df.columns:
    df_orig = train_df[['cleaned_text']].rename(columns={'cleaned_text': 'text'})
    df_aug = train_df[['aug_aeda']].rename(columns={'aug_aeda': 'text'})
    df_combined = pd.concat([df_orig, df_aug], ignore_index=True)
    print(f"\n合并后的训练集大小: {len(df_combined)} 行")
    print(df_combined.sample(5))
else:
    print("无法构建合并数据，缺少增强列或标签列。")


合并后的训练集大小: 13840 行
                                                    text
13067  i think my : boo went and found . , another bo...
11179  just wanna be ; proud : of myself for ; ; swit...
7061   ; ; me ? encanta ver a aludidos que tratan de ...
10219  no puedo, : tengo que ver al campeón , atlas s...
11248  this ? comment is fantastic, the , , follow up...


In [31]:
# 假设train_df已经有cleaned_text和aug_aeda，以及你所有的原始字段
import pandas as pd

# 1. 构建原始数据部分
df_cleaned = train_df.copy()
df_cleaned['text'] = df_cleaned['cleaned_text']
df_cleaned['source'] = 'cleaned'

# 2. 构建AEDA增强部分
df_aeda = train_df.copy()
df_aeda['text'] = df_aeda['aug_aeda']
df_aeda['source'] = 'aeda'

# 3. 拼接
df_all = pd.concat([df_cleaned, df_aeda], ignore_index=True)

# 4. 可选：只保留想要的列（如text放最前，其余原始字段跟上）
# columns = ['text', 'source'] + [col for col in train_df.columns if col not in ['cleaned_text', 'aug_aeda']]
# df_all = df_all[columns]

# 5. 保存
output_file = "EXIST2025_cleaned_plus_AEDA_fullinfo.csv"
df_all.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"已保存至 {output_file}，共 {len(df_all)} 行（原始 + 增强），所有字段已保留。")

已保存至 EXIST2025_cleaned_plus_AEDA_fullinfo.csv，共 13840 行（原始 + 增强），所有字段已保留。
